# ResNet50 Transfer Learning + Fine-Tuning — ISIC 2019 Skin Lesion Classification

## Experiment 03

**Architecture:** ResNet50  
**Training Strategy:** Transfer Learning + Fine-Tuning  
**Pretrained Weights:** ImageNet  
**Data Augmentation:** No  
**Dataset:** ISIC 2019  
**Number of Classes:** 8  
**Model Input Size:** 224 × 224  
**Cached Image Size:** 256 × 256  
**Framework:** PyTorch  
**Hardware:** Apple Silicon  
**Compute Device:** MPS  

---

## Objective

Train ResNet50 for 8-class skin-lesion classification using ImageNet
pretrained weights.

Unlike the previous ResNet50-from-scratch experiment, this model starts
with visual features learned from the ImageNet dataset.

The experiment will be performed in two stages:

### Stage 1 — Transfer Learning / Feature Extraction

- Load ResNet50 with pretrained ImageNet weights.
- Replace the original ImageNet classification layer with a new
  8-class classification layer.
- Freeze the pretrained convolutional backbone.
- Train only the new classification layer.

### Stage 2 — Fine-Tuning

- Unfreeze pretrained ResNet50 layers.
- Continue training using a smaller learning rate.
- Allow the pretrained features to adapt to skin-lesion images.

---

## Experimental Control

To keep the comparison fair with previous experiments, we will reuse:

- The same lesion-aware train/validation/test splits
- The same 256 × 256 cached images
- The same 224 × 224 model input size
- The same ImageNet normalization
- The same weighted CrossEntropyLoss
- The same evaluation metrics
- The same test set

No data augmentation will be used in this experiment.

This allows us to compare:

1. ResNet50 trained from scratch
2. DenseNet121 trained from scratch
3. ResNet50 using transfer learning and fine-tuning

In [ ]:
# ============================================================
# Imports
# ============================================================

from pathlib import Path
import random
import time
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from torchvision import transforms

from torchvision.models import (
    resnet50,
    ResNet50_Weights
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# ============================================================
# Environment Information
# ============================================================

print("=" * 60)
print("ENVIRONMENT INFORMATION")
print("=" * 60)

print("PyTorch version :", torch.__version__)
print("NumPy version   :", np.__version__)
print("Pandas version  :", pd.__version__)

print("\nLibraries imported successfully.")

In [ ]:
# ============================================================
# Project Paths
# ============================================================

PROJECT_DIR = Path.cwd()

DATA_DIR = (
    PROJECT_DIR
    / "dataset"
)

SPLIT_DIR = (
    DATA_DIR
    / "splits"
)

TRAIN_CSV = (
    SPLIT_DIR
    / "train.csv"
)

VAL_CSV = (
    SPLIT_DIR
    / "val.csv"
)

TEST_CSV = (
    SPLIT_DIR
    / "test.csv"
)

MODEL_DIR = (
    PROJECT_DIR
    / "saved_models"
)

RESULTS_DIR = (
    PROJECT_DIR
    / "results"
)


# ============================================================
# Create Output Directories
# ============================================================

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# Experiment Configuration
# ============================================================

NUM_CLASSES = 8

IMAGE_SIZE = 224

CACHE_SIZE = 256

RANDOM_SEED = 42


# ============================================================
# Display Configuration
# ============================================================

print("=" * 60)
print("PROJECT CONFIGURATION")
print("=" * 60)

print(
    "Project directory :",
    PROJECT_DIR
)

print(
    "Data directory    :",
    DATA_DIR
)

print(
    "Split directory   :",
    SPLIT_DIR
)

print(
    "\nTrain CSV exists :",
    TRAIN_CSV.exists()
)

print(
    "Val CSV exists   :",
    VAL_CSV.exists()
)

print(
    "Test CSV exists  :",
    TEST_CSV.exists()
)

print(
    "\nModel directory   :",
    MODEL_DIR
)

print(
    "Results directory :",
    RESULTS_DIR
)

print(
    "\nNumber of classes :",
    NUM_CLASSES
)

print(
    "Model image size  :",
    IMAGE_SIZE
)

print(
    "Cache image size  :",
    CACHE_SIZE
)

print(
    "Random seed       :",
    RANDOM_SEED
)

In [ ]:
# ============================================================
# Reproducibility Configuration
# ============================================================

random.seed(RANDOM_SEED)

np.random.seed(
    RANDOM_SEED
)

torch.manual_seed(
    RANDOM_SEED
)


# ============================================================
# Display Configuration
# ============================================================

print("=" * 60)
print("REPRODUCIBILITY CONFIGURATION")
print("=" * 60)

print(
    "Python random seed :",
    RANDOM_SEED
)

print(
    "NumPy random seed  :",
    RANDOM_SEED
)

print(
    "PyTorch random seed:",
    RANDOM_SEED
)

print(
    "\nRandom seeds configured successfully."
)

In [ ]:
# ============================================================
# Device Configuration
# ============================================================

if torch.backends.mps.is_available():

    DEVICE = torch.device(
        "mps"
    )

elif torch.cuda.is_available():

    DEVICE = torch.device(
        "cuda"
    )

else:

    DEVICE = torch.device(
        "cpu"
    )


# ============================================================
# Display Device Information
# ============================================================

print("=" * 60)
print("DEVICE CONFIGURATION")
print("=" * 60)

print(
    "Selected device :",
    DEVICE
)


if DEVICE.type == "mps":

    print(
        "Apple Metal Performance Shaders "
        "(MPS) is available."
    )

elif DEVICE.type == "cuda":

    print(
        "CUDA GPU is available."
    )

    print(
        "GPU name:",
        torch.cuda.get_device_name(0)
    )

else:

    print(
        "Using CPU."
    )

In [ ]:
# ============================================================
# Load Permanent Dataset Splits
# ============================================================

train_df = pd.read_csv(
    TRAIN_CSV
)

val_df = pd.read_csv(
    VAL_CSV
)

test_df = pd.read_csv(
    TEST_CSV
)


# ============================================================
# Display Split Information
# ============================================================

print("=" * 60)
print("DATASET SPLITS")
print("=" * 60)

print(
    f"Training samples   : "
    f"{len(train_df):,}"
)

print(
    f"Validation samples : "
    f"{len(val_df):,}"
)

print(
    f"Test samples       : "
    f"{len(test_df):,}"
)

print(
    f"Total samples      : "
    f"{len(train_df) + len(val_df) + len(test_df):,}"
)


# ============================================================
# Inspect Columns
# ============================================================

print(
    "\nTraining columns:"
)

print(
    train_df.columns.tolist()
)


# ============================================================
# Preview First Row
# ============================================================

print(
    "\nFirst training row:"
)

display(
    train_df.head(1)
)

In [ ]:
# ============================================================
# Build Class Information Table
# ============================================================

class_info = (
    train_df[
        [
            "class_index",
            "class_code",
            "class_name"
        ]
    ]
    .drop_duplicates()
    .sort_values("class_index")
    .reset_index(drop=True)
)


# ============================================================
# Display Class Information
# ============================================================

print("=" * 60)
print("CLASS INFORMATION")
print("=" * 60)

for _, row in class_info.iterrows():

    print(
        f"{int(row['class_index'])} "
        f"- {row['class_code']:5s} "
        f"- {row['class_name']}"
    )


print(
    "\nNumber of unique classes:",
    len(class_info)
)

In [ ]:
# ============================================================
# Verify Existing Image Cache
# ============================================================

CACHE_DIR = (
    DATA_DIR
    / f"ISIC_2019_{CACHE_SIZE}"
)

cached_images = list(
    CACHE_DIR.glob("*.jpg")
)

expected_images = (
    len(train_df)
    + len(val_df)
    + len(test_df)
)


# ============================================================
# Display Cache Information
# ============================================================

print("=" * 60)
print("IMAGE CACHE VERIFICATION")
print("=" * 60)

print(
    "Cache directory :",
    CACHE_DIR
)

print(
    "Cache exists    :",
    CACHE_DIR.exists()
)

print(
    f"\nCached images   : "
    f"{len(cached_images):,}"
)

print(
    f"Expected images : "
    f"{expected_images:,}"
)


# ============================================================
# Verify Cache Completeness
# ============================================================

if (
    CACHE_DIR.exists()
    and len(cached_images) == expected_images
):

    print(
        "\n✓ Existing 256x256 image cache "
        "is complete and ready to use."
    )

else:

    print(
        "\n⚠ Cache verification failed."
    )

    print(
        "Check the cache folder before "
        "continuing."
    )

In [ ]:
# ============================================================
# Add Cached Image Paths
# ============================================================

def add_cached_path(dataframe):

    dataframe = dataframe.copy()

    dataframe["cached_image_path"] = (
        dataframe["image_path"]
        .apply(
            lambda path: str(
                CACHE_DIR
                / Path(path).name
            )
        )
    )

    return dataframe


# ============================================================
# Apply to All Splits
# ============================================================

train_df = add_cached_path(
    train_df
)

val_df = add_cached_path(
    val_df
)

test_df = add_cached_path(
    test_df
)


# ============================================================
# Verify Paths
# ============================================================

print("=" * 60)
print("CACHED IMAGE PATHS")
print("=" * 60)

print(
    "Example training cached path:"
)

print(
    train_df.iloc[0][
        "cached_image_path"
    ]
)


print(
    "\nExample file exists:",
    Path(
        train_df.iloc[0][
            "cached_image_path"
        ]
    ).exists()
)


# ============================================================
# Check Missing Cached Files
# ============================================================

print(
    "\nMissing cached training images:",
    sum(
        not Path(path).exists()
        for path in train_df[
            "cached_image_path"
        ]
    )
)

print(
    "Missing cached validation images:",
    sum(
        not Path(path).exists()
        for path in val_df[
            "cached_image_path"
        ]
    )
)

print(
    "Missing cached test images:",
    sum(
        not Path(path).exists()
        for path in test_df[
            "cached_image_path"
        ]
    )
)

In [ ]:
# ============================================================
# ImageNet Normalization Values
# ============================================================

IMAGE_MEAN = [
    0.485,
    0.456,
    0.406
]

IMAGE_STD = [
    0.229,
    0.224,
    0.225
]


# ============================================================
# Base Preprocessing Pipeline
# ============================================================

base_transform = transforms.Compose([
    transforms.Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGE_MEAN,
        std=IMAGE_STD
    )
])


# ============================================================
# Display Preprocessing Configuration
# ============================================================

print("=" * 60)
print("IMAGE PREPROCESSING PIPELINE")
print("=" * 60)

print(
    "Input cache size :",
    f"{CACHE_SIZE} x {CACHE_SIZE}"
)

print(
    "Model input size :",
    f"{IMAGE_SIZE} x {IMAGE_SIZE}"
)

print(
    "\nNormalization mean:",
    IMAGE_MEAN
)

print(
    "Normalization std :",
    IMAGE_STD
)

print(
    "\nPreprocessing pipeline:"
)

print(
    base_transform
)

In [ ]:
# ============================================================
# Custom ISIC Dataset
# ============================================================

class ISICDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.dataframe = (
            dataframe
            .reset_index(drop=True)
        )

        self.transform = transform


    def __len__(self):

        return len(
            self.dataframe
        )


    def __getitem__(
        self,
        index
    ):

        row = self.dataframe.iloc[
            index
        ]

        image_path = row[
            "cached_image_path"
        ]

        label = int(
            row[
                "class_index"
            ]
        )


        # ====================================================
        # Load Image
        # ====================================================

        with Image.open(
            image_path
        ) as image:

            image = image.convert(
                "RGB"
            )


            # ================================================
            # Apply Transform
            # ================================================

            if self.transform is not None:

                image = self.transform(
                    image
                )


        return (
            image,
            label
        )


# ============================================================
# Confirmation
# ============================================================

print("=" * 60)
print("CUSTOM DATASET")
print("=" * 60)

print(
    "ISICDataset class created successfully."
)

In [ ]:
# ============================================================
# Create PyTorch Dataset Objects
# ============================================================

train_dataset = ISICDataset(
    dataframe=train_df,
    transform=base_transform
)

val_dataset = ISICDataset(
    dataframe=val_df,
    transform=base_transform
)

test_dataset = ISICDataset(
    dataframe=test_df,
    transform=base_transform
)


# ============================================================
# Display Dataset Sizes
# ============================================================

print("=" * 60)
print("PYTORCH DATASETS")
print("=" * 60)

print(
    f"Training samples   : "
    f"{len(train_dataset):,}"
)

print(
    f"Validation samples : "
    f"{len(val_dataset):,}"
)

print(
    f"Test samples       : "
    f"{len(test_dataset):,}"
)

print(
    f"Total samples      : "
    f"{len(train_dataset) + len(val_dataset) + len(test_dataset):,}"
)

print(
    "\nDataset objects created successfully."
)

In [ ]:
# ============================================================
# DataLoader Configuration
# ============================================================

BATCH_SIZE = 32

NUM_WORKERS = 0


# ============================================================
# Display Configuration
# ============================================================

print("=" * 60)
print("DATALOADER CONFIGURATION")
print("=" * 60)

print(
    "Batch size  :",
    BATCH_SIZE
)

print(
    "Num workers :",
    NUM_WORKERS
)

print(
    "\nDataLoader configuration ready."
)

In [ ]:
# ============================================================
# Create DataLoaders
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)


# ============================================================
# Display DataLoader Information
# ============================================================

print("=" * 60)
print("DATALOADERS")
print("=" * 60)

print(
    f"Training batches   : "
    f"{len(train_loader):,}"
)

print(
    f"Validation batches : "
    f"{len(val_loader):,}"
)

print(
    f"Test batches       : "
    f"{len(test_loader):,}"
)

print(
    "\nDataLoaders created successfully."
)

In [ ]:
# ============================================================
# Verify One Full Batch
# ============================================================

images, labels = next(
    iter(train_loader)
)


# ============================================================
# Display Batch Information
# ============================================================

print("=" * 60)
print("BATCH VERIFICATION")
print("=" * 60)

print(
    "Images batch shape :",
    images.shape
)

print(
    "Labels batch shape :",
    labels.shape
)

print(
    "Images dtype       :",
    images.dtype
)

print(
    "Labels dtype       :",
    labels.dtype
)

print(
    "\nFirst 8 labels:"
)

print(
    labels[:8]
)


# ============================================================
# Assertions
# ============================================================

assert images.shape[1:] == (
    3,
    IMAGE_SIZE,
    IMAGE_SIZE
)

assert (
    images.dtype
    == torch.float32
)

assert (
    labels.dtype
    == torch.int64
)

assert (
    len(images)
    <= BATCH_SIZE
)

assert (
    len(labels)
    == len(images)
)


print(
    "\n✓ Batch verification successful."
)

In [ ]:
# ============================================================
# Load ImageNet Pretrained ResNet50
# ============================================================

weights = ResNet50_Weights.DEFAULT

model = resnet50(
    weights=weights
)


# ============================================================
# Display Model Information
# ============================================================

print("=" * 60)
print("RESNET50 PRETRAINED MODEL")
print("=" * 60)

print(
    "Architecture       : ResNet50"
)

print(
    "Pretrained weights : ImageNet"
)

print(
    "Weight source      : ResNet50_Weights.DEFAULT"
)

print(
    "Training strategy  : Transfer Learning + Fine-Tuning"
)

print(
    "\nPretrained ResNet50 loaded successfully."
)

In [ ]:
# ============================================================
# Inspect Original ResNet50 Classifier
# ============================================================

original_in_features = (
    model.fc.in_features
)

original_out_features = (
    model.fc.out_features
)


print("=" * 60)
print("ORIGINAL RESNET50 CLASSIFIER")
print("=" * 60)

print(
    "Input features  :",
    original_in_features
)

print(
    "Output classes  :",
    original_out_features
)

print(
    "\nOriginal classifier:"
)

print(
    model.fc
)


# ============================================================
# Replace 1000-Class ImageNet Classifier
# with 8-Class ISIC Classifier
# ============================================================

model.fc = nn.Linear(
    in_features=original_in_features,
    out_features=NUM_CLASSES
)


# ============================================================
# Display New Classifier
# ============================================================

print(
    "\n"
    + "=" * 60
)

print(
    "NEW ISIC CLASSIFIER"
)

print(
    "=" * 60
)

print(
    "Input features :",
    original_in_features
)

print(
    "Output classes :",
    NUM_CLASSES
)

print(
    "\nNew classifier:"
)

print(
    model.fc
)

In [ ]:
# ============================================================
# Freeze All Pretrained ResNet50 Parameters
# ============================================================

for parameter in model.parameters():

    parameter.requires_grad = False


# ============================================================
# Unfreeze the New Classification Layer
# ============================================================

for parameter in model.fc.parameters():

    parameter.requires_grad = True


# ============================================================
# Count Parameters
# ============================================================

total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

frozen_params = (
    total_params
    - trainable_params
)


# ============================================================
# Display Freeze Configuration
# ============================================================

print("=" * 60)
print("STAGE 1 - FEATURE EXTRACTION")
print("=" * 60)

print(
    "Pretrained backbone : Frozen"
)

print(
    "New classifier      : Trainable"
)

print(
    f"\nTotal parameters     : "
    f"{total_params:,}"
)

print(
    f"Trainable parameters : "
    f"{trainable_params:,}"
)

print(
    f"Frozen parameters    : "
    f"{frozen_params:,}"
)


# ============================================================
# Verify Classifier Parameters
# ============================================================

print(
    "\nClassifier parameter status:"
)

for name, parameter in model.fc.named_parameters():

    print(
        f"{name:10s} -> "
        f"requires_grad = {parameter.requires_grad}"
    )

In [ ]:
# ============================================================
# Move Model to Selected Device
# ============================================================

model = model.to(
    DEVICE
)


# ============================================================
# Display Device Information
# ============================================================

print("=" * 60)
print("MODEL DEVICE")
print("=" * 60)

print(
    "ResNet50 moved to device:",
    DEVICE
)

print(
    "Model parameter device   :",
    next(
        model.parameters()
    ).device
)

print(
    "\n✓ Pretrained ResNet50 is ready on the selected device."
)

In [ ]:
# ============================================================
# Verify Frozen and Trainable Parameters
# ============================================================

frozen_parameter_names = []

trainable_parameter_names = []


for name, parameter in model.named_parameters():

    if parameter.requires_grad:

        trainable_parameter_names.append(
            name
        )

    else:

        frozen_parameter_names.append(
            name
        )


# ============================================================
# Display Summary
# ============================================================

print("=" * 70)
print("PARAMETER TRAINING STATUS")
print("=" * 70)

print(
    "Trainable parameter tensors :",
    len(trainable_parameter_names)
)

print(
    "Frozen parameter tensors    :",
    len(frozen_parameter_names)
)


# ============================================================
# Display Trainable Parameters
# ============================================================

print(
    "\nTrainable parameters:"
)

for name in trainable_parameter_names:

    print(
        "-",
        name
    )


# ============================================================
# Final Verification
# ============================================================

expected_trainable = {
    "fc.weight",
    "fc.bias"
}

actual_trainable = set(
    trainable_parameter_names
)


assert (
    actual_trainable
    == expected_trainable
)


print(
    "\n✓ Only the new fully connected classifier "
    "is trainable in Stage 1."
)

In [ ]:
# ============================================================
# Calculate Class Counts
# ============================================================

class_counts = (
    train_df["class_index"]
    .value_counts()
    .sort_index()
)

total_training_samples = len(
    train_df
)


# ============================================================
# Calculate Class Weights
# Formula:
# weight_c = N / (K * n_c)
# ============================================================

class_weights = (
    total_training_samples
    / (
        NUM_CLASSES
        * class_counts.values
    )
)


# ============================================================
# Convert to PyTorch Tensor
# ============================================================

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE
)


# ============================================================
# Display Results
# ============================================================

print("=" * 60)
print("CLASS WEIGHTS")
print("=" * 60)

print(
    "\nTraining class counts:"
)

print(
    class_counts
)

print(
    "\nCalculated class weights:"
)


for index, weight in enumerate(
    class_weights
):

    print(
        f"{index} - "
        f"{class_info.iloc[index]['class_code']:5s} - "
        f"{class_info.iloc[index]['class_name']}: "
        f"{weight:.4f}"
    )


print(
    "\nClass weight tensor:"
)

print(
    class_weights_tensor
)

In [ ]:
# ============================================================
# Weighted CrossEntropy Loss
# ============================================================

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)


# ============================================================
# Display Loss Configuration
# ============================================================

print("=" * 60)
print("LOSS FUNCTION")
print("=" * 60)

print(
    "Loss function : Weighted CrossEntropyLoss"
)

print(
    "Class weights : Enabled"
)

print(
    "Device        :",
    DEVICE
)

print(
    "\nCriterion:"
)

print(
    criterion
)

In [ ]:
# ============================================================
# Stage 1 Optimizer Configuration
# ============================================================

STAGE1_LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4


# ============================================================
# Optimizer
# Only Trainable Parameters Are Included
# ============================================================

optimizer = torch.optim.AdamW(
    filter(
        lambda parameter: parameter.requires_grad,
        model.parameters()
    ),
    lr=STAGE1_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# Display Optimizer Configuration
# ============================================================

print("=" * 60)
print("STAGE 1 OPTIMIZER")
print("=" * 60)

print(
    "Optimizer       : AdamW"
)

print(
    "Learning rate   :",
    STAGE1_LEARNING_RATE
)

print(
    "Weight decay    :",
    WEIGHT_DECAY
)

print(
    "Training target : New fc classifier only"
)

print(
    "\nOptimizer created successfully."
)

In [ ]:
# ============================================================
# Stage 1 Learning Rate Scheduler
# ============================================================

stage1_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)


# ============================================================
# Display Scheduler Configuration
# ============================================================

print("=" * 60)
print("STAGE 1 LEARNING RATE SCHEDULER")
print("=" * 60)

print(
    "Scheduler : ReduceLROnPlateau"
)

print(
    "Mode      : min"
)

print(
    "Factor    : 0.5"
)

print(
    "Patience  : 2"
)

print(
    "\nStage 1 scheduler created successfully."
)

In [ ]:
# ============================================================
# Stage 1 Training Configuration
# ============================================================

STAGE1_NUM_EPOCHS = 10

STAGE1_EARLY_STOPPING_PATIENCE = 3


# ============================================================
# Stage 1 Checkpoint and History Paths
# ============================================================

STAGE1_BEST_MODEL_PATH = (
    MODEL_DIR
    / "resnet50_transfer_stage1_best.pth"
)

STAGE1_HISTORY_PATH = (
    RESULTS_DIR
    / "resnet50_transfer_stage1_history.csv"
)


# ============================================================
# Display Configuration
# ============================================================

print("=" * 60)
print("STAGE 1 TRAINING CONFIGURATION")
print("=" * 60)

print(
    "Maximum epochs          :",
    STAGE1_NUM_EPOCHS
)

print(
    "Early stopping patience :",
    STAGE1_EARLY_STOPPING_PATIENCE
)

print(
    "\nBest model path:"
)

print(
    STAGE1_BEST_MODEL_PATH
)

print(
    "\nTraining history path:"
)

print(
    STAGE1_HISTORY_PATH
)

In [ ]:
# ============================================================
# Train One Epoch
# ============================================================

def train_one_epoch(
    model,
    data_loader,
    criterion,
    optimizer,
    device
):

    model.train()

    running_loss = 0.0
    running_correct = 0
    running_samples = 0


    for images, labels in data_loader:

        # ----------------------------------------------------
        # Move Batch to Device
        # ----------------------------------------------------

        images = images.to(
            device
        )

        labels = labels.to(
            device
        )


        # ----------------------------------------------------
        # Clear Previous Gradients
        # ----------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )


        # ----------------------------------------------------
        # Forward Pass
        # ----------------------------------------------------

        outputs = model(
            images
        )


        # ----------------------------------------------------
        # Calculate Loss
        # ----------------------------------------------------

        loss = criterion(
            outputs,
            labels
        )


        # ----------------------------------------------------
        # Backpropagation
        # ----------------------------------------------------

        loss.backward()


        # ----------------------------------------------------
        # Update Trainable Parameters
        # ----------------------------------------------------

        optimizer.step()


        # ----------------------------------------------------
        # Accumulate Loss
        # ----------------------------------------------------

        batch_size = images.size(
            0
        )

        running_loss += (
            loss.item()
            * batch_size
        )


        # ----------------------------------------------------
        # Predictions
        # ----------------------------------------------------

        predictions = torch.argmax(
            outputs,
            dim=1
        )


        # ----------------------------------------------------
        # Count Correct Predictions
        # ----------------------------------------------------

        running_correct += (
            predictions
            == labels
        ).sum().item()

        running_samples += (
            batch_size
        )


    # ========================================================
    # Calculate Epoch Metrics
    # ========================================================

    epoch_loss = (
        running_loss
        / running_samples
    )

    epoch_accuracy = (
        running_correct
        / running_samples
    )


    return (
        epoch_loss,
        epoch_accuracy
    )


# ============================================================
# Confirmation
# ============================================================

print("=" * 60)
print("TRAINING FUNCTION")
print("=" * 60)

print(
    "train_one_epoch() created successfully."
)

In [ ]:
# ============================================================
# Validate One Epoch
# ============================================================

def validate_one_epoch(
    model,
    data_loader,
    criterion,
    device
):

    # --------------------------------------------------------
    # Evaluation Mode
    # --------------------------------------------------------

    model.eval()

    running_loss = 0.0
    running_correct = 0
    running_samples = 0

    all_labels = []
    all_predictions = []


    # ========================================================
    # Disable Gradient Calculation
    # ========================================================

    with torch.no_grad():

        for images, labels in data_loader:

            # ------------------------------------------------
            # Move Batch to Device
            # ------------------------------------------------

            images = images.to(
                device
            )

            labels = labels.to(
                device
            )


            # ------------------------------------------------
            # Forward Pass
            # ------------------------------------------------

            outputs = model(
                images
            )


            # ------------------------------------------------
            # Calculate Validation Loss
            # ------------------------------------------------

            loss = criterion(
                outputs,
                labels
            )


            # ------------------------------------------------
            # Accumulate Loss
            # ------------------------------------------------

            batch_size = images.size(
                0
            )

            running_loss += (
                loss.item()
                * batch_size
            )


            # ------------------------------------------------
            # Predictions
            # ------------------------------------------------

            predictions = torch.argmax(
                outputs,
                dim=1
            )


            # ------------------------------------------------
            # Count Correct Predictions
            # ------------------------------------------------

            running_correct += (
                predictions
                == labels
            ).sum().item()

            running_samples += (
                batch_size
            )


            # ------------------------------------------------
            # Store Labels and Predictions
            # ------------------------------------------------

            all_labels.extend(
                labels
                .cpu()
                .numpy()
            )

            all_predictions.extend(
                predictions
                .cpu()
                .numpy()
            )


    # ========================================================
    # Calculate Validation Metrics
    # ========================================================

    epoch_loss = (
        running_loss
        / running_samples
    )

    epoch_accuracy = (
        running_correct
        / running_samples
    )

    epoch_macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro"
    )


    return (
        epoch_loss,
        epoch_accuracy,
        epoch_macro_f1
    )


# ============================================================
# Confirmation
# ============================================================

print("=" * 60)
print("VALIDATION FUNCTION")
print("=" * 60)

print(
    "validate_one_epoch() created successfully."
)

In [ ]:
# ============================================================
# Stage 1 Training History
# ============================================================

stage1_history = {
    "epoch": [],
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_macro_f1": [],
    "learning_rate": [],
    "epoch_time_seconds": []
}


# ============================================================
# Best Model Tracking
# ============================================================

stage1_best_val_f1 = -1.0

stage1_epochs_without_improvement = 0


# ============================================================
# Display Initial State
# ============================================================

print("=" * 60)
print("STAGE 1 HISTORY INITIALIZATION")
print("=" * 60)

print(
    "Best validation Macro F1    :",
    stage1_best_val_f1
)

print(
    "Epochs without improvement :",
    stage1_epochs_without_improvement
)

print(
    "\nHistory fields:"
)

for key in stage1_history.keys():

    print(
        "-",
        key
    )

print(
    "\nStage 1 tracking initialized successfully."
)

In [ ]:
# ============================================================
# Stage 1 One-Batch Forward + Backward Sanity Check
# ============================================================

# Use evaluation mode during this sanity check so pretrained
# BatchNorm running statistics are not modified.
model.eval()


# ============================================================
# Get One Training Batch
# ============================================================

sanity_images, sanity_labels = next(
    iter(train_loader)
)

sanity_images = sanity_images.to(
    DEVICE
)

sanity_labels = sanity_labels.to(
    DEVICE
)


# ============================================================
# Clear Existing Gradients
# ============================================================

optimizer.zero_grad(
    set_to_none=True
)


# ============================================================
# Forward Pass
# ============================================================

sanity_outputs = model(
    sanity_images
)


# ============================================================
# Calculate Loss
# ============================================================

sanity_loss = criterion(
    sanity_outputs,
    sanity_labels
)


# ============================================================
# Backward Pass
# ============================================================

sanity_loss.backward()


# ============================================================
# Verify Gradient Flow
# ============================================================

classifier_gradients = []

backbone_gradients = []


for name, parameter in model.named_parameters():

    if parameter.grad is not None:

        if name.startswith("fc."):

            classifier_gradients.append(
                name
            )

        else:

            backbone_gradients.append(
                name
            )


# ============================================================
# Display Results
# ============================================================

print("=" * 70)
print("STAGE 1 FORWARD + BACKWARD SANITY CHECK")
print("=" * 70)

print(
    "Input shape  :",
    sanity_images.shape
)

print(
    "Output shape :",
    sanity_outputs.shape
)

print(
    "Loss         :",
    f"{sanity_loss.item():.4f}"
)

print(
    "\nClassifier parameters receiving gradients:"
)

for name in classifier_gradients:

    print(
        "-",
        name
    )


print(
    "\nBackbone parameters receiving gradients:",
    len(backbone_gradients)
)


# ============================================================
# Assertions
# ============================================================

assert sanity_outputs.shape == (
    sanity_images.size(0),
    NUM_CLASSES
)

assert torch.isfinite(
    sanity_loss
).item()

assert set(
    classifier_gradients
) == {
    "fc.weight",
    "fc.bias"
}

assert len(
    backbone_gradients
) == 0


# ============================================================
# Clear Sanity-Check Gradients
# ============================================================

optimizer.zero_grad(
    set_to_none=True
)


print(
    "\n✓ Forward pass successful."
)

print(
    "✓ Loss calculation successful."
)

print(
    "✓ Backpropagation successful."
)

print(
    "✓ Gradients are reaching only the new classifier."
)

print(
    "✓ Frozen backbone received no gradients."
)

In [ ]:
# ============================================================
# Stage 1 BatchNorm Handling
# ============================================================

def set_batchnorm_eval(module):

    if isinstance(
        module,
        nn.modules.batchnorm._BatchNorm
    ):

        module.eval()


# ============================================================
# Redefine Stage 1 Training Function
# with Frozen BatchNorm Statistics
# ============================================================

def train_one_epoch(
    model,
    data_loader,
    criterion,
    optimizer,
    device
):

    # --------------------------------------------------------
    # Put model into training mode
    # --------------------------------------------------------

    model.train()


    # --------------------------------------------------------
    # Keep BatchNorm layers in evaluation mode
    # during frozen-backbone Stage 1
    # --------------------------------------------------------

    model.apply(
        set_batchnorm_eval
    )


    running_loss = 0.0
    running_correct = 0
    running_samples = 0


    for images, labels in data_loader:

        # ----------------------------------------------------
        # Move Batch to Device
        # ----------------------------------------------------

        images = images.to(
            device
        )

        labels = labels.to(
            device
        )


        # ----------------------------------------------------
        # Clear Previous Gradients
        # ----------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )


        # ----------------------------------------------------
        # Forward Pass
        # ----------------------------------------------------

        outputs = model(
            images
        )


        # ----------------------------------------------------
        # Calculate Loss
        # ----------------------------------------------------

        loss = criterion(
            outputs,
            labels
        )


        # ----------------------------------------------------
        # Backward Pass
        # ----------------------------------------------------

        loss.backward()


        # ----------------------------------------------------
        # Update Trainable Parameters
        # ----------------------------------------------------

        optimizer.step()


        # ----------------------------------------------------
        # Batch Statistics
        # ----------------------------------------------------

        batch_size = images.size(
            0
        )

        running_loss += (
            loss.item()
            * batch_size
        )


        predictions = torch.argmax(
            outputs,
            dim=1
        )


        running_correct += (
            predictions
            == labels
        ).sum().item()

        running_samples += (
            batch_size
        )


    # ========================================================
    # Epoch Metrics
    # ========================================================

    epoch_loss = (
        running_loss
        / running_samples
    )

    epoch_accuracy = (
        running_correct
        / running_samples
    )


    return (
        epoch_loss,
        epoch_accuracy
    )


# ============================================================
# Verify BatchNorm Status
# ============================================================

model.train()

model.apply(
    set_batchnorm_eval
)

batchnorm_layers = [
    module
    for module in model.modules()
    if isinstance(
        module,
        nn.modules.batchnorm._BatchNorm
    )
]

batchnorm_training_states = [
    module.training
    for module in batchnorm_layers
]


print("=" * 70)
print("STAGE 1 BATCHNORM CONFIGURATION")
print("=" * 70)

print(
    "BatchNorm layers found :",
    len(batchnorm_layers)
)

print(
    "BatchNorm layers in training mode:",
    sum(batchnorm_training_states)
)

print(
    "BatchNorm layers in eval mode:",
    len(batchnorm_layers)
    - sum(batchnorm_training_states)
)

assert not any(
    batchnorm_training_states
)

print(
    "\n✓ Frozen backbone BatchNorm statistics "
    "will remain fixed during Stage 1."
)

In [ ]:
# ============================================================
# Save Stage 1 Experiment Configuration
# ============================================================

stage1_config = {
    "model_name": "ResNet50",

    "experiment": "transfer_learning_stage1",

    "training_strategy": "feature_extraction",

    "pretrained_weights": "ImageNet",

    "data_augmentation": False,

    "num_classes": NUM_CLASSES,

    "image_size": IMAGE_SIZE,

    "cache_size": CACHE_SIZE,

    "batch_size": BATCH_SIZE,

    "num_workers": NUM_WORKERS,

    "optimizer": "AdamW",

    "learning_rate": STAGE1_LEARNING_RATE,

    "weight_decay": WEIGHT_DECAY,

    "scheduler": "ReduceLROnPlateau",

    "scheduler_mode": "min",

    "scheduler_factor": 0.5,

    "scheduler_patience": 2,

    "max_epochs": STAGE1_NUM_EPOCHS,

    "early_stopping_patience": STAGE1_EARLY_STOPPING_PATIENCE,

    "loss_function": "Weighted CrossEntropyLoss",

    "backbone_frozen": True,

    "batchnorm_frozen": True,

    "trainable_layer": "fc",

    "random_seed": RANDOM_SEED,

    "device": str(DEVICE)
}


# ============================================================
# Configuration File Path
# ============================================================

STAGE1_CONFIG_PATH = (
    RESULTS_DIR
    / "resnet50_transfer_stage1_config.json"
)


# ============================================================
# Save Configuration
# ============================================================

with open(
    STAGE1_CONFIG_PATH,
    "w"
) as file:

    json.dump(
        stage1_config,
        file,
        indent=4
    )


# ============================================================
# Display Saved Configuration
# ============================================================

print("=" * 70)
print("STAGE 1 EXPERIMENT CONFIGURATION")
print("=" * 70)

print(
    "Configuration saved to:"
)

print(
    STAGE1_CONFIG_PATH
)

print(
    "\nStage 1 configuration:"
)

for key, value in stage1_config.items():

    print(
        f"{key}: {value}"
    )

print(
    "\n✓ Stage 1 experiment configuration "
    "saved successfully."
)

In [ ]:
# ============================================================
# Stage 1 Training Loop
# ============================================================

stage1_training_start_time = time.time()


for epoch in range(
    1,
    STAGE1_NUM_EPOCHS + 1
):

    # ========================================================
    # Start Epoch Timer
    # ========================================================

    epoch_start_time = time.time()


    # ========================================================
    # Training Phase
    # ========================================================

    train_loss, train_accuracy = train_one_epoch(
        model=model,
        data_loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE
    )


    # ========================================================
    # Validation Phase
    # ========================================================

    (
        val_loss,
        val_accuracy,
        val_macro_f1
    ) = validate_one_epoch(
        model=model,
        data_loader=val_loader,
        criterion=criterion,
        device=DEVICE
    )


    # ========================================================
    # Learning Rate Scheduler
    # ========================================================

    stage1_scheduler.step(
        val_loss
    )


    # ========================================================
    # Current Learning Rate
    # ========================================================

    current_lr = (
        optimizer
        .param_groups[0]["lr"]
    )


    # ========================================================
    # Epoch Time
    # ========================================================

    epoch_time = (
        time.time()
        - epoch_start_time
    )


    # ========================================================
    # Store History
    # ========================================================

    stage1_history["epoch"].append(
        epoch
    )

    stage1_history["train_loss"].append(
        train_loss
    )

    stage1_history["train_accuracy"].append(
        train_accuracy
    )

    stage1_history["val_loss"].append(
        val_loss
    )

    stage1_history["val_accuracy"].append(
        val_accuracy
    )

    stage1_history["val_macro_f1"].append(
        val_macro_f1
    )

    stage1_history["learning_rate"].append(
        current_lr
    )

    stage1_history["epoch_time_seconds"].append(
        epoch_time
    )


    # ========================================================
    # Save History After Every Epoch
    # ========================================================

    stage1_history_df = pd.DataFrame(
        stage1_history
    )

    stage1_history_df.to_csv(
        STAGE1_HISTORY_PATH,
        index=False
    )


    # ========================================================
    # Display Epoch Results
    # ========================================================

    print(
        "\n"
        + "=" * 75
    )

    print(
        f"STAGE 1 - Epoch "
        f"{epoch}/{STAGE1_NUM_EPOCHS}"
    )

    print(
        "=" * 75
    )

    print(
        f"Train Loss      : "
        f"{train_loss:.4f}"
    )

    print(
        f"Train Accuracy  : "
        f"{train_accuracy * 100:.2f}%"
    )

    print(
        f"Validation Loss : "
        f"{val_loss:.4f}"
    )

    print(
        f"Validation Acc  : "
        f"{val_accuracy * 100:.2f}%"
    )

    print(
        f"Validation F1   : "
        f"{val_macro_f1:.4f}"
    )

    print(
        f"Learning Rate   : "
        f"{current_lr:.8f}"
    )

    print(
        f"Epoch Time      : "
        f"{epoch_time / 60:.2f} minutes"
    )


    # ========================================================
    # Save Best Stage 1 Checkpoint
    # ========================================================

    if val_macro_f1 > stage1_best_val_f1:

        stage1_best_val_f1 = (
            val_macro_f1
        )

        stage1_epochs_without_improvement = 0


        torch.save(
            model.state_dict(),
            STAGE1_BEST_MODEL_PATH
        )


        print(
            "\n✓ New best Stage 1 model saved."
        )

        print(
            f"Best Validation Macro F1: "
            f"{stage1_best_val_f1:.4f}"
        )


    else:

        stage1_epochs_without_improvement += 1


        print(
            "\nNo new best model."
        )

        print(
            "Epochs without improvement:",
            stage1_epochs_without_improvement
        )


    # ========================================================
    # Early Stopping
    # ========================================================

    if (
        stage1_epochs_without_improvement
        >= STAGE1_EARLY_STOPPING_PATIENCE
    ):

        print(
            "\n"
            + "=" * 75
        )

        print(
            "STAGE 1 EARLY STOPPING TRIGGERED"
        )

        print(
            "=" * 75
        )

        print(
            f"No improvement in Validation Macro F1 "
            f"for {STAGE1_EARLY_STOPPING_PATIENCE} "
            f"consecutive epochs."
        )

        break


# ============================================================
# Total Stage 1 Training Time
# ============================================================

stage1_total_training_time = (
    time.time()
    - stage1_training_start_time
)

stage1_epochs_completed = len(
    stage1_history["epoch"]
)


# ============================================================
# Final Stage 1 Summary
# ============================================================

print(
    "\n"
    + "=" * 75
)

print(
    "STAGE 1 TRAINING COMPLETE"
)

print(
    "=" * 75
)

print(
    "Epochs completed          :",
    stage1_epochs_completed
)

print(
    f"Best Validation Macro F1 : "
    f"{stage1_best_val_f1:.4f}"
)

print(
    f"Total Training Time      : "
    f"{stage1_total_training_time / 60:.2f} minutes"
)

print(
    "\nBest Stage 1 model saved to:"
)

print(
    STAGE1_BEST_MODEL_PATH
)

print(
    "\nStage 1 history saved to:"
)

print(
    STAGE1_HISTORY_PATH
)

In [ ]:
# ============================================================
# Load Stage 1 Training History
# ============================================================

stage1_history_df = pd.read_csv(
    STAGE1_HISTORY_PATH
)


# ============================================================
# Identify Best Stage 1 Epoch
# ============================================================

stage1_best_epoch_index = (
    stage1_history_df["val_macro_f1"]
    .idxmax()
)

stage1_best_epoch_row = (
    stage1_history_df
    .loc[stage1_best_epoch_index]
)

stage1_best_epoch = int(
    stage1_best_epoch_row["epoch"]
)


# ============================================================
# Display Stage 1 Summary
# ============================================================

print("=" * 75)
print("STAGE 1 TRAINING HISTORY SUMMARY")
print("=" * 75)

print(
    "Epochs completed :",
    len(stage1_history_df)
)

print(
    "Best epoch       :",
    stage1_best_epoch
)

print(
    f"Best Val Macro F1: "
    f"{stage1_best_epoch_row['val_macro_f1']:.4f}"
)

print(
    f"Val Accuracy     : "
    f"{stage1_best_epoch_row['val_accuracy'] * 100:.2f}%"
)

print(
    f"Val Loss         : "
    f"{stage1_best_epoch_row['val_loss']:.4f}"
)

print(
    f"Train Accuracy   : "
    f"{stage1_best_epoch_row['train_accuracy'] * 100:.2f}%"
)

print(
    f"Train Loss       : "
    f"{stage1_best_epoch_row['train_loss']:.4f}"
)

print(
    f"Learning Rate    : "
    f"{stage1_best_epoch_row['learning_rate']:.8f}"
)

print(
    f"Epoch Time       : "
    f"{stage1_best_epoch_row['epoch_time_seconds'] / 60:.2f} minutes"
)


# ============================================================
# Display Complete Stage 1 History
# ============================================================

print(
    "\nComplete Stage 1 history:"
)

display(
    stage1_history_df
)

In [ ]:
# ============================================================
# Load Best Stage 1 Checkpoint
# ============================================================

print("=" * 70)
print("LOADING BEST STAGE 1 CHECKPOINT")
print("=" * 70)


# ============================================================
# Verify Checkpoint Exists
# ============================================================

if not STAGE1_BEST_MODEL_PATH.exists():

    raise FileNotFoundError(
        f"Stage 1 checkpoint not found:\n"
        f"{STAGE1_BEST_MODEL_PATH}"
    )


# ============================================================
# Load Saved Weights
# ============================================================

stage1_best_state = torch.load(
    STAGE1_BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=True
)


# ============================================================
# Restore Model Weights
# ============================================================

model.load_state_dict(
    stage1_best_state
)

model = model.to(
    DEVICE
)

model.eval()


# ============================================================
# Display Restored Checkpoint Information
# ============================================================

print(
    "Checkpoint loaded successfully."
)

print(
    "\nBest Stage 1 checkpoint:"
)

print(
    STAGE1_BEST_MODEL_PATH
)

print(
    "\nBest Stage 1 epoch:",
    stage1_best_epoch
)

print(
    f"Best Stage 1 validation Macro F1: "
    f"{stage1_best_epoch_row['val_macro_f1']:.4f}"
)

print(
    "\nModel device:",
    next(
        model.parameters()
    ).device
)

print(
    "\n✓ Best Stage 1 model restored successfully."
)

In [ ]:
# ============================================================
# Stage 2 — Unfreeze ResNet50 Backbone
# ============================================================

for parameter in model.parameters():

    parameter.requires_grad = True


# ============================================================
# Count Parameters
# ============================================================

total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

frozen_params = (
    total_params
    - trainable_params
)


# ============================================================
# Display Fine-Tuning Configuration
# ============================================================

print("=" * 70)
print("STAGE 2 - FINE-TUNING CONFIGURATION")
print("=" * 70)

print(
    "Pretrained backbone : Trainable"
)

print(
    "Classifier          : Trainable"
)

print(
    f"\nTotal parameters     : "
    f"{total_params:,}"
)

print(
    f"Trainable parameters : "
    f"{trainable_params:,}"
)

print(
    f"Frozen parameters    : "
    f"{frozen_params:,}"
)


# ============================================================
# Verification
# ============================================================

assert frozen_params == 0

assert all(
    parameter.requires_grad
    for parameter in model.parameters()
)


print(
    "\n✓ Entire ResNet50 network is now trainable."
)

print(
    "✓ Model is ready for Stage 2 fine-tuning."
)

In [ ]:
# ============================================================
# Stage 2 Fine-Tuning Training Function
# ============================================================

def train_one_epoch_finetune(
    model,
    data_loader,
    criterion,
    optimizer,
    device
):

    # --------------------------------------------------------
    # Full training mode
    # BatchNorm now updates normally because the backbone
    # is unfrozen during Stage 2.
    # --------------------------------------------------------

    model.train()


    running_loss = 0.0
    running_correct = 0
    running_samples = 0


    for images, labels in data_loader:

        # ----------------------------------------------------
        # Move Batch to Device
        # ----------------------------------------------------

        images = images.to(
            device
        )

        labels = labels.to(
            device
        )


        # ----------------------------------------------------
        # Clear Previous Gradients
        # ----------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )


        # ----------------------------------------------------
        # Forward Pass
        # ----------------------------------------------------

        outputs = model(
            images
        )


        # ----------------------------------------------------
        # Calculate Loss
        # ----------------------------------------------------

        loss = criterion(
            outputs,
            labels
        )


        # ----------------------------------------------------
        # Backpropagation
        # ----------------------------------------------------

        loss.backward()


        # ----------------------------------------------------
        # Update All Trainable Parameters
        # ----------------------------------------------------

        optimizer.step()


        # ----------------------------------------------------
        # Accumulate Loss
        # ----------------------------------------------------

        batch_size = images.size(
            0
        )

        running_loss += (
            loss.item()
            * batch_size
        )


        # ----------------------------------------------------
        # Predictions
        # ----------------------------------------------------

        predictions = torch.argmax(
            outputs,
            dim=1
        )


        # ----------------------------------------------------
        # Count Correct Predictions
        # ----------------------------------------------------

        running_correct += (
            predictions
            == labels
        ).sum().item()

        running_samples += (
            batch_size
        )


    # ========================================================
    # Epoch Metrics
    # ========================================================

    epoch_loss = (
        running_loss
        / running_samples
    )

    epoch_accuracy = (
        running_correct
        / running_samples
    )


    return (
        epoch_loss,
        epoch_accuracy
    )


# ============================================================
# Confirmation
# ============================================================

print("=" * 70)
print("STAGE 2 TRAINING FUNCTION")
print("=" * 70)

print(
    "train_one_epoch_finetune() created successfully."
)

print(
    "\nStage 2 behavior:"
)

print(
    "- Entire ResNet50 is trainable"
)

print(
    "- BatchNorm layers operate in training mode"
)

print(
    "- All unfrozen parameters can receive gradients"
)

In [ ]:
# ============================================================
# Stage 2 Fine-Tuning Learning Rates
# ============================================================

BACKBONE_LEARNING_RATE = 1e-5

CLASSIFIER_LEARNING_RATE = 1e-4


# ============================================================
# Separate Backbone and Classifier Parameters
# ============================================================

backbone_parameters = [
    parameter
    for name, parameter in model.named_parameters()
    if not name.startswith("fc.")
]

classifier_parameters = list(
    model.fc.parameters()
)


# ============================================================
# Stage 2 Optimizer
# ============================================================

stage2_optimizer = torch.optim.AdamW(
    [
        {
            "params": backbone_parameters,
            "lr": BACKBONE_LEARNING_RATE
        },
        {
            "params": classifier_parameters,
            "lr": CLASSIFIER_LEARNING_RATE
        }
    ],
    weight_decay=WEIGHT_DECAY
)


# ============================================================
# Display Optimizer Configuration
# ============================================================

print("=" * 70)
print("STAGE 2 FINE-TUNING OPTIMIZER")
print("=" * 70)

print(
    "Optimizer                : AdamW"
)

print(
    "Backbone learning rate   :",
    BACKBONE_LEARNING_RATE
)

print(
    "Classifier learning rate :",
    CLASSIFIER_LEARNING_RATE
)

print(
    "Weight decay             :",
    WEIGHT_DECAY
)


# ============================================================
# Verify Parameter Groups
# ============================================================

print(
    "\nOptimizer parameter groups:"
)

for index, group in enumerate(
    stage2_optimizer.param_groups
):

    print(
        f"Group {index + 1}: "
        f"LR = {group['lr']}"
    )


print(
    "\n✓ Stage 2 optimizer created successfully."
)

In [ ]:
# ============================================================
# Stage 2 Learning Rate Scheduler
# ============================================================

stage2_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    stage2_optimizer,
    mode="min",
    factor=0.5,
    patience=2
)


# ============================================================
# Display Scheduler Configuration
# ============================================================

print("=" * 70)
print("STAGE 2 LEARNING RATE SCHEDULER")
print("=" * 70)

print(
    "Scheduler : ReduceLROnPlateau"
)

print(
    "Mode      : min"
)

print(
    "Factor    : 0.5"
)

print(
    "Patience  : 2"
)


# ============================================================
# Display Initial Learning Rates
# ============================================================

print(
    "\nInitial learning rates:"
)

for index, group in enumerate(
    stage2_optimizer.param_groups
):

    print(
        f"Group {index + 1}: "
        f"{group['lr']}"
    )


print(
    "\n✓ Stage 2 scheduler created successfully."
)

In [ ]:
# ============================================================
# Stage 2 Training Configuration
# ============================================================

STAGE2_NUM_EPOCHS = 15

STAGE2_EARLY_STOPPING_PATIENCE = 4


# ============================================================
# Stage 2 Checkpoint and History Paths
# ============================================================

STAGE2_BEST_MODEL_PATH = (
    MODEL_DIR
    / "resnet50_transfer_stage2_best.pth"
)

STAGE2_HISTORY_PATH = (
    RESULTS_DIR
    / "resnet50_transfer_stage2_history.csv"
)


# ============================================================
# Display Configuration
# ============================================================

print("=" * 70)
print("STAGE 2 TRAINING CONFIGURATION")
print("=" * 70)

print(
    "Maximum epochs          :",
    STAGE2_NUM_EPOCHS
)

print(
    "Early stopping patience :",
    STAGE2_EARLY_STOPPING_PATIENCE
)

print(
    "\nBest model path:"
)

print(
    STAGE2_BEST_MODEL_PATH
)

print(
    "\nTraining history path:"
)

print(
    STAGE2_HISTORY_PATH
)

In [ ]:
# ============================================================
# Stage 2 Training History
# ============================================================

stage2_history = {
    "epoch": [],
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_macro_f1": [],
    "backbone_learning_rate": [],
    "classifier_learning_rate": [],
    "epoch_time_seconds": []
}


# ============================================================
# Best Model Tracking
# ============================================================

stage2_best_val_f1 = -1.0

stage2_epochs_without_improvement = 0


# ============================================================
# Display Initial State
# ============================================================

print("=" * 70)
print("STAGE 2 HISTORY INITIALIZATION")
print("=" * 70)

print(
    "Best validation Macro F1    :",
    stage2_best_val_f1
)

print(
    "Epochs without improvement :",
    stage2_epochs_without_improvement
)

print(
    "\nHistory fields:"
)

for key in stage2_history.keys():

    print(
        "-",
        key
    )

print(
    "\nStage 2 tracking initialized successfully."
)

In [ ]:
# ============================================================
# Stage 2 One-Batch Forward + Backward Sanity Check
# ============================================================

# Evaluation mode prevents BatchNorm running statistics
# from being changed during this temporary sanity check.
# Gradients are still calculated normally.
model.eval()


# ============================================================
# Get One Training Batch
# ============================================================

sanity_images, sanity_labels = next(
    iter(train_loader)
)

sanity_images = sanity_images.to(
    DEVICE
)

sanity_labels = sanity_labels.to(
    DEVICE
)


# ============================================================
# Clear Existing Gradients
# ============================================================

stage2_optimizer.zero_grad(
    set_to_none=True
)


# ============================================================
# Forward Pass
# ============================================================

sanity_outputs = model(
    sanity_images
)


# ============================================================
# Calculate Loss
# ============================================================

sanity_loss = criterion(
    sanity_outputs,
    sanity_labels
)


# ============================================================
# Backward Pass
# ============================================================

sanity_loss.backward()


# ============================================================
# Inspect Gradient Flow
# ============================================================

backbone_gradient_parameters = []

classifier_gradient_parameters = []


for name, parameter in model.named_parameters():

    if parameter.grad is not None:

        if name.startswith("fc."):

            classifier_gradient_parameters.append(
                name
            )

        else:

            backbone_gradient_parameters.append(
                name
            )


# ============================================================
# Display Results
# ============================================================

print("=" * 75)
print("STAGE 2 FORWARD + BACKWARD SANITY CHECK")
print("=" * 75)

print(
    "Input shape  :",
    sanity_images.shape
)

print(
    "Output shape :",
    sanity_outputs.shape
)

print(
    "Loss         :",
    f"{sanity_loss.item():.4f}"
)


print(
    "\nBackbone parameters receiving gradients:",
    len(backbone_gradient_parameters)
)

print(
    "Classifier parameters receiving gradients:",
    len(classifier_gradient_parameters)
)


print(
    "\nClassifier gradient parameters:"
)

for name in classifier_gradient_parameters:

    print(
        "-",
        name
    )


# ============================================================
# Assertions
# ============================================================

assert sanity_outputs.shape == (
    sanity_images.size(0),
    NUM_CLASSES
)

assert torch.isfinite(
    sanity_loss
).item()

assert len(
    backbone_gradient_parameters
) > 0

assert set(
    classifier_gradient_parameters
) == {
    "fc.weight",
    "fc.bias"
}


# ============================================================
# Clear Sanity-Check Gradients
# ============================================================

stage2_optimizer.zero_grad(
    set_to_none=True
)


print(
    "\n✓ Forward pass successful."
)

print(
    "✓ Loss calculation successful."
)

print(
    "✓ Backpropagation successful."
)

print(
    "✓ Gradients are reaching the pretrained backbone."
)

print(
    "✓ Gradients are reaching the classifier."
)

print(
    "✓ Stage 2 fine-tuning setup is ready."
)

In [ ]:
# ============================================================
# Save Stage 2 Fine-Tuning Configuration
# ============================================================

stage2_config = {
    "model_name": "ResNet50",

    "experiment": "transfer_learning_stage2",

    "training_strategy": "fine_tuning",

    "pretrained_weights": "ImageNet",

    "data_augmentation": False,

    "num_classes": NUM_CLASSES,

    "image_size": IMAGE_SIZE,

    "cache_size": CACHE_SIZE,

    "batch_size": BATCH_SIZE,

    "num_workers": NUM_WORKERS,

    "optimizer": "AdamW",

    "backbone_learning_rate": BACKBONE_LEARNING_RATE,

    "classifier_learning_rate": CLASSIFIER_LEARNING_RATE,

    "weight_decay": WEIGHT_DECAY,

    "scheduler": "ReduceLROnPlateau",

    "scheduler_mode": "min",

    "scheduler_factor": 0.5,

    "scheduler_patience": 2,

    "max_epochs": STAGE2_NUM_EPOCHS,

    "early_stopping_patience": STAGE2_EARLY_STOPPING_PATIENCE,

    "loss_function": "Weighted CrossEntropyLoss",

    "backbone_frozen": False,

    "batchnorm_frozen": False,

    "random_seed": RANDOM_SEED,

    "device": str(DEVICE),

    "starting_checkpoint": str(
        STAGE1_BEST_MODEL_PATH
    )
}


# ============================================================
# Configuration File Path
# ============================================================

STAGE2_CONFIG_PATH = (
    RESULTS_DIR
    / "resnet50_transfer_stage2_config.json"
)


# ============================================================
# Save Configuration
# ============================================================

with open(
    STAGE2_CONFIG_PATH,
    "w"
) as file:

    json.dump(
        stage2_config,
        file,
        indent=4
    )


# ============================================================
# Display Saved Configuration
# ============================================================

print("=" * 75)
print("STAGE 2 EXPERIMENT CONFIGURATION")
print("=" * 75)

print(
    "Configuration saved to:"
)

print(
    STAGE2_CONFIG_PATH
)

print(
    "\nStage 2 configuration:"
)

for key, value in stage2_config.items():

    print(
        f"{key}: {value}"
    )

print(
    "\n✓ Stage 2 fine-tuning configuration "
    "saved successfully."
)

In [ ]:
# ============================================================
# Stage 2 Fine-Tuning Training Loop
# ============================================================

stage2_training_start_time = time.time()


for epoch in range(
    1,
    STAGE2_NUM_EPOCHS + 1
):

    # ========================================================
    # Start Epoch Timer
    # ========================================================

    epoch_start_time = time.time()


    # ========================================================
    # Training Phase
    # ========================================================

    train_loss, train_accuracy = train_one_epoch_finetune(
        model=model,
        data_loader=train_loader,
        criterion=criterion,
        optimizer=stage2_optimizer,
        device=DEVICE
    )


    # ========================================================
    # Validation Phase
    # ========================================================

    (
        val_loss,
        val_accuracy,
        val_macro_f1
    ) = validate_one_epoch(
        model=model,
        data_loader=val_loader,
        criterion=criterion,
        device=DEVICE
    )


    # ========================================================
    # Learning Rate Scheduler
    # ========================================================

    stage2_scheduler.step(
        val_loss
    )


    # ========================================================
    # Current Learning Rates
    # ========================================================

    current_backbone_lr = (
        stage2_optimizer
        .param_groups[0]["lr"]
    )

    current_classifier_lr = (
        stage2_optimizer
        .param_groups[1]["lr"]
    )


    # ========================================================
    # Epoch Time
    # ========================================================

    epoch_time = (
        time.time()
        - epoch_start_time
    )


    # ========================================================
    # Store Training History
    # ========================================================

    stage2_history["epoch"].append(
        epoch
    )

    stage2_history["train_loss"].append(
        train_loss
    )

    stage2_history["train_accuracy"].append(
        train_accuracy
    )

    stage2_history["val_loss"].append(
        val_loss
    )

    stage2_history["val_accuracy"].append(
        val_accuracy
    )

    stage2_history["val_macro_f1"].append(
        val_macro_f1
    )

    stage2_history[
        "backbone_learning_rate"
    ].append(
        current_backbone_lr
    )

    stage2_history[
        "classifier_learning_rate"
    ].append(
        current_classifier_lr
    )

    stage2_history[
        "epoch_time_seconds"
    ].append(
        epoch_time
    )


    # ========================================================
    # Save History After Every Epoch
    # ========================================================

    stage2_history_df = pd.DataFrame(
        stage2_history
    )

    stage2_history_df.to_csv(
        STAGE2_HISTORY_PATH,
        index=False
    )


    # ========================================================
    # Display Epoch Results
    # ========================================================

    print(
        "\n"
        + "=" * 80
    )

    print(
        f"STAGE 2 - Epoch "
        f"{epoch}/{STAGE2_NUM_EPOCHS}"
    )

    print(
        "=" * 80
    )

    print(
        f"Train Loss          : "
        f"{train_loss:.4f}"
    )

    print(
        f"Train Accuracy      : "
        f"{train_accuracy * 100:.2f}%"
    )

    print(
        f"Validation Loss     : "
        f"{val_loss:.4f}"
    )

    print(
        f"Validation Accuracy : "
        f"{val_accuracy * 100:.2f}%"
    )

    print(
        f"Validation Macro F1 : "
        f"{val_macro_f1:.4f}"
    )

    print(
        f"Backbone LR         : "
        f"{current_backbone_lr:.8f}"
    )

    print(
        f"Classifier LR       : "
        f"{current_classifier_lr:.8f}"
    )

    print(
        f"Epoch Time          : "
        f"{epoch_time / 60:.2f} minutes"
    )


    # ========================================================
    # Save Best Stage 2 Checkpoint
    # ========================================================

    if val_macro_f1 > stage2_best_val_f1:

        stage2_best_val_f1 = (
            val_macro_f1
        )

        stage2_epochs_without_improvement = 0


        torch.save(
            model.state_dict(),
            STAGE2_BEST_MODEL_PATH
        )


        print(
            "\n✓ New best Stage 2 model saved."
        )

        print(
            f"Best Validation Macro F1: "
            f"{stage2_best_val_f1:.4f}"
        )


    else:

        stage2_epochs_without_improvement += 1


        print(
            "\nNo new best model."
        )

        print(
            "Epochs without improvement:",
            stage2_epochs_without_improvement
        )


    # ========================================================
    # Early Stopping
    # ========================================================

    if (
        stage2_epochs_without_improvement
        >= STAGE2_EARLY_STOPPING_PATIENCE
    ):

        print(
            "\n"
            + "=" * 80
        )

        print(
            "STAGE 2 EARLY STOPPING TRIGGERED"
        )

        print(
            "=" * 80
        )

        print(
            f"No improvement in Validation Macro F1 "
            f"for {STAGE2_EARLY_STOPPING_PATIENCE} "
            f"consecutive epochs."
        )

        break


# ============================================================
# Total Stage 2 Training Time
# ============================================================

stage2_total_training_time = (
    time.time()
    - stage2_training_start_time
)

stage2_epochs_completed = len(
    stage2_history["epoch"]
)


# ============================================================
# Final Stage 2 Summary
# ============================================================

print(
    "\n"
    + "=" * 80
)

print(
    "STAGE 2 FINE-TUNING COMPLETE"
)

print(
    "=" * 80
)

print(
    "Epochs completed          :",
    stage2_epochs_completed
)

print(
    f"Best Validation Macro F1 : "
    f"{stage2_best_val_f1:.4f}"
)

print(
    f"Total Training Time      : "
    f"{stage2_total_training_time / 60:.2f} minutes"
)

print(
    "\nBest Stage 2 model saved to:"
)

print(
    STAGE2_BEST_MODEL_PATH
)

print(
    "\nStage 2 history saved to:"
)

print(
    STAGE2_HISTORY_PATH
)

In [ ]:
# ============================================================
# Load Stage 2 Training History
# ============================================================

stage2_history_df = pd.read_csv(
    STAGE2_HISTORY_PATH
)


# ============================================================
# Identify Best Stage 2 Epoch
# ============================================================

stage2_best_epoch_index = (
    stage2_history_df["val_macro_f1"]
    .idxmax()
)

stage2_best_epoch_row = (
    stage2_history_df
    .loc[stage2_best_epoch_index]
)

stage2_best_epoch = int(
    stage2_best_epoch_row["epoch"]
)


# ============================================================
# Display Stage 2 Summary
# ============================================================

print("=" * 80)
print("STAGE 2 TRAINING HISTORY SUMMARY")
print("=" * 80)

print(
    "Epochs completed :",
    len(stage2_history_df)
)

print(
    "Best epoch       :",
    stage2_best_epoch
)

print(
    f"Best Val Macro F1: "
    f"{stage2_best_epoch_row['val_macro_f1']:.4f}"
)

print(
    f"Val Accuracy     : "
    f"{stage2_best_epoch_row['val_accuracy'] * 100:.2f}%"
)

print(
    f"Val Loss         : "
    f"{stage2_best_epoch_row['val_loss']:.4f}"
)

print(
    f"Train Accuracy   : "
    f"{stage2_best_epoch_row['train_accuracy'] * 100:.2f}%"
)

print(
    f"Train Loss       : "
    f"{stage2_best_epoch_row['train_loss']:.4f}"
)

print(
    f"Backbone LR      : "
    f"{stage2_best_epoch_row['backbone_learning_rate']:.8f}"
)

print(
    f"Classifier LR    : "
    f"{stage2_best_epoch_row['classifier_learning_rate']:.8f}"
)

print(
    f"Epoch Time       : "
    f"{stage2_best_epoch_row['epoch_time_seconds'] / 60:.2f} minutes"
)


# ============================================================
# Display Complete Stage 2 History
# ============================================================

print(
    "\nComplete Stage 2 history:"
)

display(
    stage2_history_df
)

In [ ]:
# ============================================================
# Plot Stage 2 Training and Validation Loss
# ============================================================

plt.figure(
    figsize=(10, 6)
)


# ============================================================
# Training Loss
# ============================================================

plt.plot(
    stage2_history_df["epoch"],
    stage2_history_df["train_loss"],
    marker="o",
    label="Training Loss"
)


# ============================================================
# Validation Loss
# ============================================================

plt.plot(
    stage2_history_df["epoch"],
    stage2_history_df["val_loss"],
    marker="o",
    label="Validation Loss"
)


# ============================================================
# Highlight Best Epoch
# ============================================================

plt.axvline(
    x=stage2_best_epoch,
    linestyle="--",
    label=f"Best Epoch = {stage2_best_epoch}"
)


# ============================================================
# Plot Formatting
# ============================================================

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Loss"
)

plt.title(
    "ResNet50 Transfer Learning - Stage 2 Loss"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# Plot Stage 2 Accuracy and Validation Macro F1
# ============================================================

plt.figure(
    figsize=(10, 6)
)


# ============================================================
# Training Accuracy
# ============================================================

plt.plot(
    stage2_history_df["epoch"],
    stage2_history_df["train_accuracy"],
    marker="o",
    label="Training Accuracy"
)


# ============================================================
# Validation Accuracy
# ============================================================

plt.plot(
    stage2_history_df["epoch"],
    stage2_history_df["val_accuracy"],
    marker="o",
    label="Validation Accuracy"
)


# ============================================================
# Validation Macro F1
# ============================================================

plt.plot(
    stage2_history_df["epoch"],
    stage2_history_df["val_macro_f1"],
    marker="o",
    label="Validation Macro F1"
)


# ============================================================
# Highlight Best Fine-Tuning Epoch
# ============================================================

plt.axvline(
    x=stage2_best_epoch,
    linestyle="--",
    label=f"Best Epoch = {stage2_best_epoch}"
)


# ============================================================
# Plot Formatting
# ============================================================

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Score"
)

plt.title(
    "ResNet50 Transfer Learning - Stage 2 Performance"
)

plt.ylim(
    0,
    1
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# Load Best Stage 2 Checkpoint
# ============================================================

print("=" * 75)
print("LOADING BEST STAGE 2 CHECKPOINT")
print("=" * 75)


# ============================================================
# Verify Checkpoint Exists
# ============================================================

if not STAGE2_BEST_MODEL_PATH.exists():

    raise FileNotFoundError(
        f"Stage 2 checkpoint not found:\n"
        f"{STAGE2_BEST_MODEL_PATH}"
    )


# ============================================================
# Load Saved State Dictionary
# ============================================================

stage2_best_state = torch.load(
    STAGE2_BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=True
)


# ============================================================
# Restore Best Fine-Tuned Weights
# ============================================================

model.load_state_dict(
    stage2_best_state
)

model = model.to(
    DEVICE
)

model.eval()


# ============================================================
# Display Restored Checkpoint Information
# ============================================================

print(
    "Checkpoint loaded successfully."
)

print(
    "\nBest Stage 2 checkpoint:"
)

print(
    STAGE2_BEST_MODEL_PATH
)

print(
    "\nBest Stage 2 epoch:",
    stage2_best_epoch
)

print(
    f"Best Stage 2 validation Macro F1: "
    f"{stage2_best_epoch_row['val_macro_f1']:.4f}"
)

print(
    "\nModel device:",
    next(
        model.parameters()
    ).device
)

print(
    "\nModel mode:",
    "Evaluation"
)

print(
    "\n✓ Best fine-tuned ResNet50 "
    "is ready for final test evaluation."
)

In [ ]:
# ============================================================
# Final Evaluation on Test Set
# ============================================================

model.eval()

test_running_loss = 0.0
test_running_samples = 0

all_test_labels = []
all_test_predictions = []
all_test_probabilities = []


# ============================================================
# Test Loop
# ============================================================

with torch.no_grad():

    for images, labels in test_loader:

        # ----------------------------------------------------
        # Move Batch to Device
        # ----------------------------------------------------

        images = images.to(
            DEVICE
        )

        labels = labels.to(
            DEVICE
        )


        # ----------------------------------------------------
        # Forward Pass
        # ----------------------------------------------------

        outputs = model(
            images
        )


        # ----------------------------------------------------
        # Calculate Test Loss
        # ----------------------------------------------------

        loss = criterion(
            outputs,
            labels
        )


        # ----------------------------------------------------
        # Convert Logits to Probabilities
        # ----------------------------------------------------

        probabilities = torch.softmax(
            outputs,
            dim=1
        )


        # ----------------------------------------------------
        # Predicted Classes
        # ----------------------------------------------------

        predictions = torch.argmax(
            outputs,
            dim=1
        )


        # ----------------------------------------------------
        # Accumulate Loss
        # ----------------------------------------------------

        batch_size = images.size(
            0
        )

        test_running_loss += (
            loss.item()
            * batch_size
        )

        test_running_samples += (
            batch_size
        )


        # ----------------------------------------------------
        # Store Results
        # ----------------------------------------------------

        all_test_labels.extend(
            labels
            .cpu()
            .numpy()
        )

        all_test_predictions.extend(
            predictions
            .cpu()
            .numpy()
        )

        all_test_probabilities.extend(
            probabilities
            .cpu()
            .numpy()
        )


# ============================================================
# Convert Results to NumPy Arrays
# ============================================================

all_test_labels = np.array(
    all_test_labels
)

all_test_predictions = np.array(
    all_test_predictions
)

all_test_probabilities = np.array(
    all_test_probabilities
)


# ============================================================
# Calculate Final Test Metrics
# ============================================================

test_loss = (
    test_running_loss
    / test_running_samples
)

test_accuracy = accuracy_score(
    all_test_labels,
    all_test_predictions
)

test_macro_f1 = f1_score(
    all_test_labels,
    all_test_predictions,
    average="macro"
)

test_weighted_f1 = f1_score(
    all_test_labels,
    all_test_predictions,
    average="weighted"
)


# ============================================================
# Display Final Test Results
# ============================================================

print("=" * 75)
print("FINAL RESNET50 TRANSFER LEARNING TEST RESULTS")
print("=" * 75)

print(
    f"Test Loss        : "
    f"{test_loss:.4f}"
)

print(
    f"Test Accuracy    : "
    f"{test_accuracy:.4f}"
)

print(
    f"Test Accuracy    : "
    f"{test_accuracy * 100:.2f}%"
)

print(
    f"Test Macro F1    : "
    f"{test_macro_f1:.4f}"
)

print(
    f"Test Weighted F1 : "
    f"{test_weighted_f1:.4f}"
)

print(
    "\nTest samples evaluated:",
    test_running_samples
)


# ============================================================
# Verification
# ============================================================

assert (
    len(all_test_labels)
    == len(test_dataset)
)

assert (
    len(all_test_predictions)
    == len(test_dataset)
)

assert all_test_probabilities.shape == (
    len(test_dataset),
    NUM_CLASSES
)


print(
    "\n✓ Final test evaluation completed successfully."
)

In [ ]:
# ============================================================
# Class Names
# ============================================================

class_names = (
    class_info
    .sort_values("class_index")["class_name"]
    .tolist()
)


# ============================================================
# Classification Report
# ============================================================

classification_report_text = classification_report(
    all_test_labels,
    all_test_predictions,
    target_names=class_names,
    digits=4
)


# ============================================================
# Display Report
# ============================================================

print("=" * 85)
print("RESNET50 TRANSFER LEARNING CLASSIFICATION REPORT")
print("=" * 85)

print(
    classification_report_text
)

In [ ]:
# ============================================================
# Convert Classification Report to Dictionary
# ============================================================

classification_report_dict = classification_report(
    all_test_labels,
    all_test_predictions,
    target_names=class_names,
    output_dict=True
)


# ============================================================
# Convert Dictionary to DataFrame
# ============================================================

report_df = pd.DataFrame(
    classification_report_dict
).transpose()


# ============================================================
# Display Per-Class Results
# ============================================================

print("=" * 85)
print("RESNET50 TRANSFER LEARNING PER-CLASS RESULTS")
print("=" * 85)

display(
    report_df
)

In [ ]:
# ============================================================
# Output Paths
# ============================================================

TEST_RESULTS_PATH = (
    RESULTS_DIR
    / "resnet50_transfer_test_results.json"
)

PER_CLASS_RESULTS_PATH = (
    RESULTS_DIR
    / "resnet50_transfer_per_class_results.csv"
)


# ============================================================
# Overall Test Results
# ============================================================

test_results = {
    "model_name": "ResNet50",

    "training_strategy": (
        "transfer_learning_and_fine_tuning"
    ),

    "pretrained_weights": "ImageNet",

    "best_stage1_epoch": int(
        stage1_best_epoch
    ),

    "best_stage1_validation_macro_f1": float(
        stage1_best_epoch_row[
            "val_macro_f1"
        ]
    ),

    "best_stage2_epoch": int(
        stage2_best_epoch
    ),

    "best_stage2_validation_macro_f1": float(
        stage2_best_epoch_row[
            "val_macro_f1"
        ]
    ),

    "test_loss": float(
        test_loss
    ),

    "test_accuracy": float(
        test_accuracy
    ),

    "test_macro_f1": float(
        test_macro_f1
    ),

    "test_weighted_f1": float(
        test_weighted_f1
    ),

    "test_samples": int(
        len(all_test_labels)
    )
}


# ============================================================
# Save Overall Metrics to JSON
# ============================================================

with open(
    TEST_RESULTS_PATH,
    "w"
) as file:

    json.dump(
        test_results,
        file,
        indent=4
    )


# ============================================================
# Save Per-Class Results to CSV
# ============================================================

report_df.to_csv(
    PER_CLASS_RESULTS_PATH
)


# ============================================================
# Display Saved Results
# ============================================================

print("=" * 80)
print("RESNET50 TRANSFER LEARNING RESULTS SAVED")
print("=" * 80)

print(
    "\nOverall test results:"
)

print(
    TEST_RESULTS_PATH
)

print(
    "\nPer-class results:"
)

print(
    PER_CLASS_RESULTS_PATH
)


print(
    "\nSaved test metrics:"
)

for key, value in test_results.items():

    print(
        f"{key}: {value}"
    )


print(
    "\n✓ Final ResNet50 transfer-learning "
    "results saved successfully."
)

In [ ]:
# ============================================================
# Raw Confusion Matrix
# ============================================================

confusion_matrix_raw = confusion_matrix(
    all_test_labels,
    all_test_predictions
)


# ============================================================
# Convert to DataFrame
# ============================================================

confusion_matrix_df = pd.DataFrame(
    confusion_matrix_raw,
    index=class_names,
    columns=class_names
)


# ============================================================
# Save Confusion Matrix
# ============================================================

CONFUSION_MATRIX_PATH = (
    RESULTS_DIR
    / "resnet50_transfer_confusion_matrix.csv"
)

confusion_matrix_df.to_csv(
    CONFUSION_MATRIX_PATH
)


# ============================================================
# Display Matrix
# ============================================================

print("=" * 85)
print("RESNET50 TRANSFER LEARNING RAW CONFUSION MATRIX")
print("=" * 85)

display(
    confusion_matrix_df
)

print(
    "\nConfusion matrix saved to:"
)

print(
    CONFUSION_MATRIX_PATH
)

print(
    "\n✓ Raw confusion matrix created and saved successfully."
)

In [ ]:
# ============================================================
# Plot Raw Confusion Matrix
# ============================================================

plt.figure(
    figsize=(12, 10)
)


# ============================================================
# Display Matrix
# ============================================================

plt.imshow(
    confusion_matrix_raw
)


# ============================================================
# Title and Axis Labels
# ============================================================

plt.title(
    "ResNet50 Transfer Learning - Raw Confusion Matrix"
)

plt.xlabel(
    "Predicted Class"
)

plt.ylabel(
    "Actual Class"
)


# ============================================================
# Class Labels
# ============================================================

plt.xticks(
    ticks=np.arange(NUM_CLASSES),
    labels=class_names,
    rotation=45,
    ha="right"
)

plt.yticks(
    ticks=np.arange(NUM_CLASSES),
    labels=class_names
)


# ============================================================
# Add Values Inside Each Cell
# ============================================================

for i in range(NUM_CLASSES):

    for j in range(NUM_CLASSES):

        plt.text(
            j,
            i,
            confusion_matrix_raw[i, j],
            ha="center",
            va="center"
        )


# ============================================================
# Add Color Scale
# ============================================================

plt.colorbar(
    label="Number of Samples"
)


# ============================================================
# Final Layout
# ============================================================

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# Normalized Confusion Matrix
# ============================================================

confusion_matrix_normalized = (
    confusion_matrix_raw
    / confusion_matrix_raw.sum(
        axis=1,
        keepdims=True
    )
)


# ============================================================
# Convert to DataFrame
# ============================================================

confusion_matrix_normalized_df = pd.DataFrame(
    confusion_matrix_normalized,
    index=class_names,
    columns=class_names
)


# ============================================================
# Save Normalized Confusion Matrix
# ============================================================

NORMALIZED_CONFUSION_MATRIX_PATH = (
    RESULTS_DIR
    / "resnet50_transfer_confusion_matrix_normalized.csv"
)

confusion_matrix_normalized_df.to_csv(
    NORMALIZED_CONFUSION_MATRIX_PATH
)


# ============================================================
# Display Matrix
# ============================================================

print("=" * 85)
print("RESNET50 TRANSFER LEARNING NORMALIZED CONFUSION MATRIX")
print("=" * 85)

display(
    confusion_matrix_normalized_df.round(4)
)

print(
    "\nNormalized confusion matrix saved to:"
)

print(
    NORMALIZED_CONFUSION_MATRIX_PATH
)

print(
    "\n✓ Normalized confusion matrix created successfully."
)

In [ ]:
# ============================================================
# Plot Normalized Confusion Matrix
# ============================================================

plt.figure(
    figsize=(12, 10)
)


# ============================================================
# Display Matrix
# ============================================================

plt.imshow(
    confusion_matrix_normalized,
    vmin=0,
    vmax=1
)


# ============================================================
# Title and Axis Labels
# ============================================================

plt.title(
    "ResNet50 Transfer Learning - Normalized Confusion Matrix"
)

plt.xlabel(
    "Predicted Class"
)

plt.ylabel(
    "Actual Class"
)


# ============================================================
# Class Labels
# ============================================================

plt.xticks(
    ticks=np.arange(NUM_CLASSES),
    labels=class_names,
    rotation=45,
    ha="right"
)

plt.yticks(
    ticks=np.arange(NUM_CLASSES),
    labels=class_names
)


# ============================================================
# Add Percentages Inside Each Cell
# ============================================================

for i in range(NUM_CLASSES):

    for j in range(NUM_CLASSES):

        value = (
            confusion_matrix_normalized[i, j]
            * 100
        )

        plt.text(
            j,
            i,
            f"{value:.1f}%",
            ha="center",
            va="center"
        )


# ============================================================
# Add Color Scale
# ============================================================

plt.colorbar(
    label="Proportion"
)


# ============================================================
# Final Layout
# ============================================================

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# Predicted Confidence Scores
# ============================================================

predicted_confidence = (
    all_test_probabilities.max(
        axis=1
    )
)


# ============================================================
# Class Indices
# ============================================================

actual_class_indices = (
    all_test_labels
)

predicted_class_indices = (
    all_test_predictions
)


# ============================================================
# Convert Class Indices to Class Names
# ============================================================

actual_class_names = [
    class_names[index]
    for index in actual_class_indices
]

predicted_class_names = [
    class_names[index]
    for index in predicted_class_indices
]


# ============================================================
# Build Prediction DataFrame
# ============================================================

test_predictions_df = pd.DataFrame({
    "actual_class_index": actual_class_indices,
    "actual_class_name": actual_class_names,

    "predicted_class_index": predicted_class_indices,
    "predicted_class_name": predicted_class_names,

    "prediction_confidence": predicted_confidence
})


# ============================================================
# Add Correct / Incorrect Column
# ============================================================

test_predictions_df["is_correct"] = (
    test_predictions_df["actual_class_index"]
    ==
    test_predictions_df["predicted_class_index"]
)


# ============================================================
# Add Probability for Each Class
# ============================================================

for class_index, class_name in enumerate(
    class_names
):

    probability_column_name = (
        f"probability_{class_index}_"
        + class_name
        .lower()
        .replace(" ", "_")
    )

    test_predictions_df[
        probability_column_name
    ] = all_test_probabilities[
        :,
        class_index
    ]


# ============================================================
# Save Predictions
# ============================================================

TEST_PREDICTIONS_PATH = (
    RESULTS_DIR
    / "resnet50_transfer_test_predictions.csv"
)

test_predictions_df.to_csv(
    TEST_PREDICTIONS_PATH,
    index=False
)


# ============================================================
# Display Summary
# ============================================================

print("=" * 85)
print("RESNET50 TRANSFER LEARNING TEST PREDICTIONS")
print("=" * 85)

print(
    "Total predictions:",
    len(test_predictions_df)
)

print(
    "Correct predictions:",
    test_predictions_df[
        "is_correct"
    ].sum()
)

print(
    "Incorrect predictions:",
    (
        ~test_predictions_df[
            "is_correct"
        ]
    ).sum()
)

print(
    "\nSaved to:"
)

print(
    TEST_PREDICTIONS_PATH
)


# ============================================================
# Preview Predictions
# ============================================================

print(
    "\nPrediction preview:"
)

display(
    test_predictions_df.head(10)
)

print(
    "\n✓ Test predictions saved successfully."
)

In [ ]:
# ============================================================
# Attach Image Information to Test Predictions
# ============================================================

test_metadata = (
    test_df
    .reset_index(drop=True)
)

test_predictions_with_paths_df = (
    test_predictions_df.copy()
)

test_predictions_with_paths_df[
    "image_name"
] = test_metadata["image_path"].apply(
    lambda path: Path(path).name
)

test_predictions_with_paths_df[
    "image_path"
] = test_metadata[
    "image_path"
]

test_predictions_with_paths_df[
    "cached_image_path"
] = test_metadata[
    "cached_image_path"
]


# ============================================================
# Extract Misclassified Predictions
# ============================================================

misclassified_df = (
    test_predictions_with_paths_df[
        test_predictions_with_paths_df[
            "is_correct"
        ] == False
    ]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# Sort by Prediction Confidence
# ============================================================

misclassified_df = (
    misclassified_df
    .sort_values(
        by="prediction_confidence",
        ascending=False
    )
    .reset_index(drop=True)
)


# ============================================================
# Save Misclassified Predictions
# ============================================================

MISCLASSIFIED_PATH = (
    RESULTS_DIR
    / "resnet50_transfer_misclassified_predictions.csv"
)

misclassified_df.to_csv(
    MISCLASSIFIED_PATH,
    index=False
)


# ============================================================
# Summary
# ============================================================

total_test_samples = len(
    test_predictions_with_paths_df
)

total_misclassified = len(
    misclassified_df
)

total_correct = (
    total_test_samples
    - total_misclassified
)


print("=" * 85)
print("RESNET50 TRANSFER LEARNING MISCLASSIFICATIONS")
print("=" * 85)

print(
    "Total test samples     :",
    total_test_samples
)

print(
    "Correct predictions    :",
    total_correct
)

print(
    "Misclassified samples  :",
    total_misclassified
)

print(
    "Misclassification rate :",
    f"{(total_misclassified / total_test_samples) * 100:.2f}%"
)


print(
    "\nSaved to:"
)

print(
    MISCLASSIFIED_PATH
)


# ============================================================
# Display Highest-Confidence Errors
# ============================================================

print(
    "\nHighest-confidence misclassifications:"
)

display(
    misclassified_df[
        [
            "image_name",
            "actual_class_name",
            "predicted_class_name",
            "prediction_confidence"
        ]
    ].head(20)
)

print(
    "\n✓ Misclassified predictions extracted "
    "and saved successfully."
)

In [ ]:
# ============================================================
# Visualize Random Misclassified Test Images
# ============================================================

NUM_IMAGES_TO_SHOW = 8


# ============================================================
# Randomly Sample Misclassified Images
# ============================================================

sample_count = min(
    NUM_IMAGES_TO_SHOW,
    len(misclassified_df)
)

sampled_errors = (
    misclassified_df
    .sample(
        n=sample_count,
        random_state=RANDOM_SEED
    )
    .reset_index(drop=True)
)


# ============================================================
# Create Figure
# ============================================================

plt.figure(
    figsize=(16, 10)
)


# ============================================================
# Display Images
# ============================================================

for index, row in sampled_errors.iterrows():

    image_path = row[
        "cached_image_path"
    ]

    actual_class = row[
        "actual_class_name"
    ]

    predicted_class = row[
        "predicted_class_name"
    ]

    confidence = row[
        "prediction_confidence"
    ]


    # --------------------------------------------------------
    # Open Image
    # --------------------------------------------------------

    with Image.open(image_path) as image:

        image = image.convert(
            "RGB"
        )


        # ----------------------------------------------------
        # Plot Image
        # ----------------------------------------------------

        plt.subplot(
            2,
            4,
            index + 1
        )

        plt.imshow(
            image
        )

        plt.axis(
            "off"
        )


        # ----------------------------------------------------
        # Add Prediction Information
        # ----------------------------------------------------

        plt.title(
            f"Actual: {actual_class}\n"
            f"Predicted: {predicted_class}\n"
            f"Confidence: {confidence * 100:.1f}%",
            fontsize=9
        )


# ============================================================
# Final Layout
# ============================================================

plt.suptitle(
    "ResNet50 Transfer Learning - Random Misclassified Test Images",
    fontsize=14
)

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# Visualize Highest-Confidence Misclassified Test Images
# ============================================================

NUM_IMAGES_TO_SHOW = 8


# ============================================================
# Select Highest-Confidence Errors
# ============================================================

top_confidence_errors = (
    misclassified_df
    .sort_values(
        by="prediction_confidence",
        ascending=False
    )
    .head(NUM_IMAGES_TO_SHOW)
    .reset_index(drop=True)
)


# ============================================================
# Create Figure
# ============================================================

plt.figure(
    figsize=(16, 10)
)


# ============================================================
# Display Images
# ============================================================

for index, row in top_confidence_errors.iterrows():

    image_path = row[
        "cached_image_path"
    ]

    actual_class = row[
        "actual_class_name"
    ]

    predicted_class = row[
        "predicted_class_name"
    ]

    confidence = row[
        "prediction_confidence"
    ]


    # --------------------------------------------------------
    # Open Image
    # --------------------------------------------------------

    with Image.open(image_path) as image:

        image = image.convert(
            "RGB"
        )


        # ----------------------------------------------------
        # Plot Image
        # ----------------------------------------------------

        plt.subplot(
            2,
            4,
            index + 1
        )

        plt.imshow(
            image
        )

        plt.axis(
            "off"
        )


        # ----------------------------------------------------
        # Add Prediction Information
        # ----------------------------------------------------

        plt.title(
            f"Actual: {actual_class}\n"
            f"Predicted: {predicted_class}\n"
            f"Confidence: {confidence * 100:.1f}%",
            fontsize=9
        )


# ============================================================
# Final Layout
# ============================================================

plt.suptitle(
    "ResNet50 Transfer Learning - Highest-Confidence Errors",
    fontsize=14
)

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# Final ResNet50 Transfer Learning Experiment Summary
# ============================================================

final_summary = {
    "Model": "ResNet50",

    "Training Strategy": (
        "Transfer Learning + Fine-Tuning"
    ),

    "Pretrained Weights": "ImageNet",

    "Data Augmentation": "No",

    "Best Stage 1 Epoch": int(
        stage1_best_epoch
    ),

    "Best Stage 1 Val Macro F1": round(
        float(
            stage1_best_epoch_row[
                "val_macro_f1"
            ]
        ),
        4
    ),

    "Best Stage 2 Epoch": int(
        stage2_best_epoch
    ),

    "Best Stage 2 Val Macro F1": round(
        float(
            stage2_best_epoch_row[
                "val_macro_f1"
            ]
        ),
        4
    ),

    "Test Loss": round(
        float(
            test_loss
        ),
        4
    ),

    "Test Accuracy (%)": round(
        float(
            test_accuracy
            * 100
        ),
        2
    ),

    "Test Macro F1": round(
        float(
            test_macro_f1
        ),
        4
    ),

    "Test Weighted F1": round(
        float(
            test_weighted_f1
        ),
        4
    ),

    "Test Samples": int(
        len(
            all_test_labels
        )
    )
}


# ============================================================
# Convert Summary to DataFrame
# ============================================================

final_summary_df = pd.DataFrame(
    list(
        final_summary.items()
    ),
    columns=[
        "Metric",
        "Value"
    ]
)


# ============================================================
# Display Summary
# ============================================================

print("=" * 85)
print("FINAL RESNET50 TRANSFER LEARNING EXPERIMENT SUMMARY")
print("=" * 85)

display(
    final_summary_df
)


# ============================================================
# Save Final Summary
# ============================================================

FINAL_SUMMARY_PATH = (
    RESULTS_DIR
    / "resnet50_transfer_final_summary.csv"
)

final_summary_df.to_csv(
    FINAL_SUMMARY_PATH,
    index=False
)


print(
    "\nFinal summary saved to:"
)

print(
    FINAL_SUMMARY_PATH
)

print(
    "\n✓ ResNet50 transfer learning + "
    "fine-tuning experiment completed successfully."
)